# Salary Prediction — Live Kafka Streaming Test (VM)

Starts the real-time prediction stream (`src/streaming/prediction_stream.py`, PLAN.md §16), publishes one test request to `salary_requests`, and reads back the matching prediction from `salary_predictions` — all from this notebook, no separate terminal needed.

**Prerequisites:** Kafka broker running, all 5 topics created, and `models/best_salary_model` already exists (i.e. `run_training_pipeline.ipynb` has been run at least once). **Run cells top to bottom.**

## 1. Path and working directory

In [ ]:
import sys, os

PROJECT_ROOT = "/home/linuxu/project"  # adjust if this VM's checkout lives elsewhere

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## 1a. Install missing Python packages (one-time)

If a later cell fails with `ModuleNotFoundError`, add `%pip install <package-name>` here and re-run from the top after restarting the kernel.

In [ ]:
%pip install python-dotenv confluent-kafka matplotlib

### If you hit a numpy/matplotlib import error (`_ARRAY_API not found`, `numpy.dtype size changed`, etc.)

Some VM conda environments have older packages (`scipy`, `numba`) pinned to numpy 1.x, so a plain `%pip install` can leave numpy/matplotlib on mismatched binary versions. Run the cell below once, then **restart the kernel** (a live kernel keeps the old binaries loaded in memory even after new ones are installed on disk) and re-run this notebook from the top.

In [ ]:
# Only run this if a later cell (matplotlib import, or a plotting cell) fails with a
# numpy/matplotlib binary-compatibility error. Then: Kernel -> Restart Kernel, and re-run
# from the top - a restart is required, installing alone does not fix an already-running
# kernel's in-memory state.
%pip install --upgrade --force-reinstall numpy matplotlib

## 2. `.env` check

In [ ]:
if not os.path.exists(".env"):
    import subprocess
    subprocess.run(["cp", ".env.example", ".env"])
    print("Created .env from .env.example.")

print(open(".env").read())

Confirm `KAFKA_BOOTSTRAP_SERVERS`, `KAFKA_REQUEST_TOPIC`, `KAFKA_PREDICTION_TOPIC`, `KAFKA_DEAD_LETTER_TOPIC` above match what you actually created on the VM (per PLAN.md §6 / the Lab3-style setup: `salary_requests`, `salary_predictions`, `salary_dead_letter`).

## 3. Spark session — reuse the existing one (Kafka already works there)

Unlike the training notebook, this one deliberately does **not** stop/recreate the session or force `spark.jars.packages`. The Kafka connector JAR isn't bundled in `$SPARK_HOME/jars` on this VM, and a freshly-created session's package-based resolution failed (`AnalysisException: Failed to find data source: kafka`) — likely no internet access for Maven to fetch it. The original working prototype (`notebooks/04_Spark_Streaming_Prediction.ipynb`) never configured this explicitly either; it just reused whatever session the kernel already had, and that session already has Kafka working (however it was originally bootstrapped). So we do the same: reuse it as-is instead of trying to build a better-configured one ourselves.

In [ ]:
from src.common.spark_session import get_spark_session
from config import settings

spark = get_spark_session(app_name="SalaryPredictionStream", with_kafka=False)
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("spark.master:", spark.sparkContext.master)
print("KAFKA_BOOTSTRAP_SERVERS  =", settings.KAFKA_BOOTSTRAP_SERVERS)
print("KAFKA_REQUEST_TOPIC      =", settings.KAFKA_REQUEST_TOPIC)
print("KAFKA_PREDICTION_TOPIC   =", settings.KAFKA_PREDICTION_TOPIC)
print("MODEL_PATH               =", settings.MODEL_PATH)
print("MODEL_PATH exists?       =", os.path.exists(settings.MODEL_PATH))

## 3a. Preflight check: is the Kafka connector actually available?

This VM is configured so a plain `pyspark`/`jupyter` launch already has the Kafka connector available - this cell is just a fast sanity check before going further. If it fails, something about the VM's Spark/Kafka connector setup has changed, or you're on a different environment than the one this was set up on; see PLAN.md §18/§23 for how it's configured.

In [ ]:
# Cheap check: constructing a readStream DataFrame resolves the data source
# immediately (it's lazy after that - no actual consuming happens here), so this
# fails fast with a clear message instead of deep inside build_streams() later.
try:
    (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
        .option("subscribe", settings.KAFKA_REQUEST_TOPIC)
        .load()
    )
    print("Kafka connector is available in this session - safe to proceed.")
except Exception as exc:
    if "Failed to find data source: kafka" in str(exc):
        print("=" * 70)
        print("KAFKA CONNECTOR NOT AVAILABLE IN THIS SESSION")
        print("=" * 70)
        print()
        print("This VM is normally configured so any pyspark/jupyter session has the")
        print("Kafka connector available automatically (PLAN.md section 18/23). If")
        print("you're seeing this, that VM-level configuration may have changed, or")
        print("you're running on a different machine than expected - check PLAN.md")
        print("before proceeding.")
        print("=" * 70)
        raise RuntimeError("Kafka connector not available - see instructions printed above.") from None
    raise

## 4. Start the prediction stream (non-blocking)

This starts both streaming queries (predictions and dead letters) and returns immediately — it does **not** block the notebook, unlike the CLI entry point (`python -m src.streaming.prediction_stream`), which awaits termination forever. You're responsible for stopping the queries yourself in the cleanup cell at the bottom.

In [ ]:
from src.streaming.prediction_stream import build_streams

prediction_query, dead_letter_query = build_streams(spark)

print("prediction_query.isActive  =", prediction_query.isActive)
print("dead_letter_query.isActive =", dead_letter_query.isActive)

## 5. Publish a test request

Same pattern as Lab3's `confluent_kafka` producer.

In [ ]:
import json
import uuid
from confluent_kafka import Producer

producer = Producer({"bootstrap.servers": settings.KAFKA_BOOTSTRAP_SERVERS})

test_request = {
    "request_id": str(uuid.uuid4()),
    "Country": "Israel",
    "Age": "25-34 years old",
    "EdLevel": "Bachelor's degree",
    "Employment": "Employed, full-time",
    "RemoteWork": "Hybrid",
    "DevType": "Developer, back-end",
    "OrgSize": "100 to 499 employees",
    "Industry": "Information Services, IT, Software Development",
    "YearsCodePro": 5,
    "LanguageHaveWorkedWith": "Python;SQL",
    "DatabaseHaveWorkedWith": "PostgreSQL",
    "PlatformHaveWorkedWith": "AWS",
}

producer.produce(
    settings.KAFKA_REQUEST_TOPIC,
    key=test_request["request_id"],
    value=json.dumps(test_request),
)
producer.flush()
print("Published request_id:", test_request["request_id"])

## 6. Wait, then check the result

Give the stream a few seconds to pick up the request and publish a prediction before checking. Re-run this cell if `matches` comes back empty the first time.

In [ ]:
import time
from pyspark.sql import functions as F

time.sleep(45)

results = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_PREDICTION_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string").alias("value"))
)

matches = [row["value"] for row in results.collect() if test_request["request_id"] in row["value"]]

if matches:
    print("Found matching prediction(s):")
    for match in matches:
        print(json.dumps(json.loads(match), indent=2))
else:
    print("No match yet for request_id", test_request["request_id"], "- wait a bit and re-run this cell.")
    print("All predictions currently on the topic:", results.count())

## 7. (Optional) Test the dead-letter path

Publishes a request with no `request_id` — should NOT appear on `salary_predictions`, and should instead show up on `salary_dead_letter`.

In [ ]:
bad_request = {"Country": "Israel", "YearsCodePro": 3}  # no request_id on purpose

producer.produce(settings.KAFKA_REQUEST_TOPIC, value=json.dumps(bad_request))
producer.flush()
print("Published a request with no request_id.")

time.sleep(45)

dead_letters = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_DEAD_LETTER_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string"))
)
dead_letters.show(truncate=False)

## 8. Predict on real historical `developer_events` & compare to real salary

Takes real historical rows from `developer_events` (published by `run_dataset_producer.ipynb`) - which carry the actual `ConvertedCompYearly` from the original survey row - scores them directly with the trained `PipelineModel`, and compares predicted vs. real salary.

This scores the model directly (`model.transform()`), not via the `salary_requests`/`salary_predictions` round trip above, so it works whether or not the streaming queries from Section 4 are still running.

**Caveat:** these events come from the same dataset the model was trained on, so this is a sanity/demo check of individual predictions, not a rigorous held-out evaluation - that's what `models/model_metrics.json`'s actual test-set RMSE/R² (from the real train/validation/test split) already is.

**Prerequisite:** `notebooks/run_dataset_producer.ipynb` must have been run at least once so `developer_events` has data.

In [ ]:
from src.common.schemas import DEVELOPER_EVENT_SCHEMA

raw_events = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_DATASET_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string").alias("raw_value"))
    .withColumn("event", F.from_json(F.col("raw_value"), DEVELOPER_EVENT_SCHEMA))
    .select("event.*")
)

# Same plausible-salary bounds data_cleaning.py applies before training (PLAN.md §10.9)
# - without this, an implausible value (e.g. a real $14/year row found in practice)
# doesn't just look odd in the table below, it dominates the mean percentage error
# (dividing by a near-zero real salary blows up pct_error even for a modest dollar gap).
events_with_salary = raw_events.filter(
    F.col("event_id").isNotNull()
    & F.col("ConvertedCompYearly").isNotNull()
    & (F.col("ConvertedCompYearly") >= settings.MIN_PLAUSIBLE_SALARY)
    & (F.col("ConvertedCompYearly") <= settings.MAX_PLAUSIBLE_SALARY)
)
print("Events with a known, plausible real salary:", events_with_salary.count())

In [ ]:
from pyspark.ml import PipelineModel

from src.common.spark_utils import reverse_log1p_predictions
from src.streaming.prediction_stream import STRING_FEATURE_COLUMNS, load_model_metadata

SAMPLE_SIZE = 30

sample_events = (
    events_with_salary.fillna("Unknown", subset=STRING_FEATURE_COLUMNS)
    .withColumnRenamed("YearsCodePro", "YearsCodeProNumeric")
    .limit(SAMPLE_SIZE)
)

metadata = load_model_metadata()
model = PipelineModel.load(settings.MODEL_PATH)
print(f"Using model: {metadata['selected_model']} (trained {metadata['trained_at']})")

scored = model.transform(sample_events)
scored = reverse_log1p_predictions(scored, log_prediction_col="log_prediction", output_col="predicted_salary")

comparison = (
    scored.select(
        F.col("event_id"),
        F.col("Country"),
        F.col("DevType"),
        F.col("ConvertedCompYearly").alias("real_salary"),
        F.round(F.col("predicted_salary"), 2).alias("predicted_salary"),
    )
    .withColumn("abs_error", F.round(F.abs(F.col("predicted_salary") - F.col("real_salary")), 2))
    .withColumn("pct_error", F.round(F.col("abs_error") / F.col("real_salary") * 100, 1))
)

# Collected once here (it's a small, bounded SAMPLE_SIZE-row result) and reused for both
# the table below and the plotting cell - no pandas needed anywhere in this section.
comparison_rows = comparison.collect()
comparison.show(comparison.count(), truncate=False)

### Real vs. predicted salary

In [ ]:
import matplotlib.pyplot as plt

real_salaries = [row["real_salary"] for row in comparison_rows]
predicted_salaries = [row["predicted_salary"] for row in comparison_rows]
abs_errors = [row["abs_error"] for row in comparison_rows]
pct_errors = [row["pct_error"] for row in comparison_rows]

plt.figure(figsize=(6, 6))
plt.scatter(real_salaries, predicted_salaries)
max_value = max(real_salaries + predicted_salaries)
plt.plot([0, max_value], [0, max_value], linestyle="--", color="gray", label="Perfect prediction")
plt.xlabel("Real salary ($)")
plt.ylabel("Predicted salary ($)")
plt.title("Real vs. Predicted Salary")
plt.legend()
plt.show()

print("Mean absolute error: $%.2f" % (sum(abs_errors) / len(abs_errors)))
print("Mean percentage error: %.1f%%" % (sum(pct_errors) / len(pct_errors)))

## 9. Cleanup — stop the streaming queries

Run this when you're done testing, so the queries don't keep running in the background.

In [ ]:
prediction_query.stop()
dead_letter_query.stop()
print("Stopped both streaming queries.")